# The version timeline: v1 min-max → v6 deployed

## Goal

Put the six versions of the engine head to head, exactly as docs/EVALUATION.md, section 12
reports it:

- **v1 — naive min-max**, hand-tuned weights (the documented baseline);
- **v2 — alpha-beta**, same weights: pure algorithmic speed, provably identical play;
- **v3 — alpha-beta + tuned weights**: the GA champion in place of the hand-tuned set;
- **v4 — quiescence, forward pruning and the opening book** (section 10);
- **v5 — the forced-win layer** (section 11): proven offense, budgeted deepening and the
  defence screen;
- **v6 — the parallel split root** (section 13), the current deployment: every root
  candidate searched as an independent pool job, a node budget in place of v5's, and the
  same book and stage-1 layer underneath.

Two questions, both at the deployment depth 3: **who wins** — every adjacent pairing of the
timeline plus the full journey v1-vs-v6, one deterministic game per color assignment through
the section-9 protocol (center opening, 60-ply cap, weights installed per search) — and
**what it costs** — the section-10 positions plus the opening reply, where the book answers
instead of a search, at depths 3 and 4.

The engines are deterministic (the v6 split is pinned to one worker here; the test suite's
V6 Test 4 pins answer invariance across workers), so every game outcome, ply count, node
count and score below reproduces docs/EVALUATION.md, sections 12.1 and 12.2 exactly; only
wall times vary. The validation cells assert all of it — a clean run of this notebook is a
validation of the section-12 tables.

## Expected behaviour

The ladder must hold: v1 and v2 split their pairing (alpha-beta changes the price of a move,
never the move — both games are the section-9 baseline mirror at ply 38); v3 beats v2 in
both colors (plies 17 and 36, the section-9 champion head-to-head); v4 beats v3 in both
colors; v4 and v5 split by color on the book line; v5 and v6 are the same player — both
games of their pairing identical move for move, the split changes the price of a move,
never the move; and v6 beats v1 in both colors on the full journey. No version ever loses
to an earlier one.

Runtime: about 10 minutes (the twelve games run one worker process each; the budget
searches of v5/v6 dominate the cost table).


In [ ]:
import os
import sys
import time
from dataclasses import dataclass
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tuning" / "default.json").is_file())
os.environ["WEIGHTS_FILE"] = str(ROOT / "tuning" / "default.json")
os.environ["V6_JOBS"] = "1"
sys.path.insert(0, str(ROOT / "tuning"))

import genetic_tuner as gt
import harness
from connect6.defines import Color, SearchStats
from connect6.opening_book import book_move, load_opening_book
from connect6.parallel import search_v6
from connect6.protocol import move2msg
from connect6.search import search_alpha_beta, search_alpha_beta_with_quiescence, search_min_max
from connect6.vcf import search_v5

DEPTH = 3
MAX_PLIES = 60


@dataclass(frozen=True, slots=True)
class Version:
    name: str
    weights: gt.Weights
    search: str
    use_book: bool


champion = harness.load_weights(ROOT / "tuning" / "winner.json")
VERSIONS = [
    Version("v1 min-max", gt.BASE, "min_max", False),
    Version("v2 alpha-beta", gt.BASE, "alpha_beta", False),
    Version("v3 alpha-beta + tuned weights", champion, "alpha_beta", False),
    Version("v4 + quiescence, pruning, book", champion, "enhanced", True),
    Version("v5 + forced-win layer", champion, "v5", True),
    Version("v6 + parallel split, budget", champion, "v6", True),
]
SEARCHES = {
    "min_max": search_min_max,
    "alpha_beta": search_alpha_beta,
    "enhanced": search_alpha_beta_with_quiescence,
    "v5": search_v5,
    "v6": search_v6,
}
BOOK = load_opening_book()

print(f"weights pinned to {os.environ['WEIGHTS_FILE']} (installed per search by the referee)")
for version in VERSIONS:
    print(f"  {version.name}: {gt.fmt(version.weights)}")


## Who is winning

Every game is one deterministic pairing; the v4/v5/v6 players consult the opening book
before searching, install their weights per search through the genetic tuner's own
`apply_weights`, and both sides share the referee from `tuning/harness.py`. The
`book hits` extras count the moves answered without a search.


In [ ]:
def think(version, board, color, depth, pre_move, stats):
    if version.use_book:
        booked = book_move(board, BOOK)
        if booked is not None:
            return booked, None
    gt.apply_weights(version.weights)
    result = SEARCHES[version.search](board, color, depth, pre_move, stats)
    return result.move, result.score


def timed_think(version, board, color, depth, pre_move, tally):
    stats = SearchStats()
    start = time.perf_counter()
    move, score = think(version, board, color, depth, pre_move, stats)
    tally.ms += (time.perf_counter() - start) * 1000
    tally.nodes += stats.node_count
    tally.moves += 1
    if version.use_book and stats.node_count == 0:
        tally.book_hits += 1
    return move, score


def make_spec(version):
    return harness.PlayerSpec(SEARCHES[version.search], version.weights, version.use_book)


pairings = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (0, 5)]
plans = []
for left, right in pairings:
    for black_first in (True, False):
        black_version = VERSIONS[left] if black_first else VERSIONS[right]
        white_version = VERSIONS[right] if black_first else VERSIONS[left]
        plans.append((black_version, white_version,
                      harness.GameSpec(make_spec(black_version), make_spec(white_version), DEPTH, MAX_PLIES)))

start = time.perf_counter()
records = harness.run_games([plan[2] for plan in plans])
print(f"== games: adjacent steps of the timeline, plus the full journey ({time.perf_counter() - start:.0f} s across twelve workers) ==")
game_rows = []
for (black_version, white_version, _), record in zip(plans, records):
    if record.winner is None:
        outcome, who = "draw", "nobody"
    else:
        outcome = f"{'black' if record.winner is Color.BLACK else 'white'} wins"
        who = black_version.name if record.winner is Color.BLACK else white_version.name
    extras = []
    if record.black.book_hits:
        extras.append(f"black book hits {record.black.book_hits}")
    if record.white.book_hits:
        extras.append(f"white book hits {record.white.book_hits}")
    print(
        f"{black_version.name} (black) vs {white_version.name} (white): "
        f"{outcome} — {who}, ply {record.plies} | "
        f"black: {record.black.moves} moves, {record.black.nodes} nodes, {record.black.ms / max(record.black.moves, 1):.0f} ms/move | "
        f"white: {record.white.moves} moves, {record.white.nodes} nodes, {record.white.ms / max(record.white.moves, 1):.0f} ms/move"
        + (" | " + ", ".join(extras) if extras else "")
    )
    game_rows.append((black_version.name, white_version.name, outcome, who, record.plies,
                      record.black.book_hits, record.white.book_hits))
print()


## Validation

The cell below re-checks every game against the documented table — outcome, winner, ply
and the book-hit pattern — and pins the two same-player mirrors: the v5-vs-v6 pairing is
literally one game played twice (identical move lists in both color assignments, the
split root answering exactly what the shared-window root answers on the book line).


In [ ]:
EXPECTED_GAMES = [
    ("v1 min-max", "v2 alpha-beta", "black wins", "v1 min-max", 38, 0, 0),
    ("v2 alpha-beta", "v1 min-max", "black wins", "v2 alpha-beta", 38, 0, 0),
    ("v2 alpha-beta", "v3 alpha-beta + tuned weights", "white wins", "v3 alpha-beta + tuned weights", 17, 0, 0),
    ("v3 alpha-beta + tuned weights", "v2 alpha-beta", "black wins", "v3 alpha-beta + tuned weights", 36, 0, 0),
    ("v3 alpha-beta + tuned weights", "v4 + quiescence, pruning, book", "white wins", "v4 + quiescence, pruning, book", 19, 0, 1),
    ("v4 + quiescence, pruning, book", "v3 alpha-beta + tuned weights", "black wins", "v4 + quiescence, pruning, book", 26, 0, 0),
    ("v4 + quiescence, pruning, book", "v5 + forced-win layer", "white wins", "v5 + forced-win layer", 37, 5, 5),
    ("v5 + forced-win layer", "v4 + quiescence, pruning, book", "white wins", "v4 + quiescence, pruning, book", 19, 5, 5),
    ("v5 + forced-win layer", "v6 + parallel split, budget", "white wins", "v6 + parallel split, budget", 19, 5, 5),
    ("v6 + parallel split, budget", "v5 + forced-win layer", "white wins", "v5 + forced-win layer", 19, 5, 5),
    ("v1 min-max", "v6 + parallel split, budget", "white wins", "v6 + parallel split, budget", 19, 0, 1),
    ("v6 + parallel split, budget", "v1 min-max", "black wins", "v6 + parallel split, budget", 30, 0, 0),
]
assert game_rows == EXPECTED_GAMES
assert records[8].moves == records[9].moves, "the v5-v6 pairing must be one game played twice"
print("all 12 games reproduce section 12.1 exactly (outcome, winner, ply, book hits)")
print("the v5-vs-v6 games are move-for-move identical in both color assignments")


## What it costs

One search per position per version, deterministic. `book` marks the position the opening
book answered without a search — v4, v5 and v6 all take the opening reply for free. The
min-max rows at depth 4 are omitted: those trees are minutes long, and the depth-3 rows
already make the point (27,931 nodes whatever the position — the naive search is
position-blind). The v5/v6 rows are **budget answers**, not fixed-depth ones: the depth is
the floor, the engine deepens while its node budget fits, and the reached depth is printed
(`d0` marks a stage-1 forced-win answer that never builds a tree).


In [ ]:
cost_rows = {}
positions = harness.documented_positions()
for depth in (DEPTH, DEPTH + 1):
    rows = [v for v in VERSIONS if depth == DEPTH or v.search != "min_max"]
    note = "" if len(rows) == len(VERSIONS) else " (min-max omitted: the naive depth-4 trees are minutes long)"
    print(f"== timing and nodes at depth {depth}{note} ==")
    print(f"{'position':34s} {'version':38s} {'nodes':>8s} {'ms':>8s} {'score':>7s}  move")
    for label, board, color, pre_move in positions:
        for version in rows:
            tally = harness.Tally()
            stats = SearchStats()
            start = time.perf_counter()
            booked = book_move(board, BOOK) if version.use_book else None
            if booked is not None:
                move, score = booked, None
                tally.ms += (time.perf_counter() - start) * 1000
            else:
                gt.apply_weights(version.weights)
                result = SEARCHES[version.search](board, color, depth, pre_move, stats)
                move, score = result.move, result.score
                tally.ms += (time.perf_counter() - start) * 1000
                tally.nodes += stats.node_count
            depth_note = f" (d{stats.depth_reached})" if version.search == "v6" and tally.nodes > 0 else ""
            print(
                f"{label:34s} {version.name:38s} {tally.nodes:8d} {tally.ms:8.1f} "
                f"{'book' if score is None else str(score):>7}  {move2msg(move)}{depth_note}"
            )
            cost_rows[(depth, label, version.name)] = (tally.nodes, score, move2msg(move))
    print()


In [ ]:
EXPECTED_COST = {
    (3, "opening reply, white to move", "v1 min-max"): (27931, -303, "IJJK"),
    (3, "opening reply, white to move", "v2 alpha-beta"): (2830, -303, "IJJK"),
    (3, "opening reply, white to move", "v3 alpha-beta + tuned weights"): (2918, -282, "IJJK"),
    (3, "opening reply, white to move", "v4 + quiescence, pruning, book"): (0, None, "IIKK"),
    (3, "opening reply, white to move", "v5 + forced-win layer"): (0, None, "IIKK"),
    (3, "opening reply, white to move", "v6 + parallel split, budget"): (0, None, "IIKK"),
    (3, "opening scatter, white to move", "v1 min-max"): (27931, -510, "KQCK"),
    (3, "opening scatter, white to move", "v2 alpha-beta"): (1182, -510, "KQCK"),
    (3, "opening scatter, white to move", "v3 alpha-beta + tuned weights"): (1593, -436, "CIIP"),
    (3, "opening scatter, white to move", "v4 + quiescence, pruning, book"): (1668, -386, "CKJP"),
    (3, "opening scatter, white to move", "v5 + forced-win layer"): (76694, -336, "IKKP"),
    (3, "opening scatter, white to move", "v6 + parallel split, budget"): (36912, -38, "CKJP"),
    (3, "midgame, white to move", "v1 min-max"): (27931, -3097, "FJII"),
    (3, "midgame, white to move", "v2 alpha-beta"): (1390, -3097, "FJII"),
    (3, "midgame, white to move", "v3 alpha-beta + tuned weights"): (1425, -2342, "FJII"),
    (3, "midgame, white to move", "v4 + quiescence, pruning, book"): (1458, -703, "BJII"),
    (3, "midgame, white to move", "v5 + forced-win layer"): (23420, -1060, "JGKI"),
    (3, "midgame, white to move", "v6 + parallel split, budget"): (52283, -617, "JGII"),
    (3, "fork attack, black to move", "v1 min-max"): (27931, 20000, "JNBJ"),
    (3, "fork attack, black to move", "v2 alpha-beta"): (78, 20000, "JNBJ"),
    (3, "fork attack, black to move", "v3 alpha-beta + tuned weights"): (78, 20000, "JNBJ"),
    (3, "fork attack, black to move", "v4 + quiescence, pruning, book"): (78, 20000, "JNBJ"),
    (3, "fork attack, black to move", "v5 + forced-win layer"): (18, 20000, "JSFJ"),
    (3, "fork attack, black to move", "v6 + parallel split, budget"): (18, 20000, "JSFJ"),
    (3, "fork completed, white to move", "v1 min-max"): (24631, 20000, "GJHJ"),
    (3, "fork completed, white to move", "v2 alpha-beta"): (351, 20000, "GJHJ"),
    (3, "fork completed, white to move", "v3 alpha-beta + tuned weights"): (347, 20000, "GJHJ"),
    (3, "fork completed, white to move", "v4 + quiescence, pruning, book"): (207, 20000, "GJHJ"),
    (3, "fork completed, white to move", "v5 + forced-win layer"): (208, 20000, "GJHJ"),
    (3, "fork completed, white to move", "v6 + parallel split, budget"): (208, 20000, "GJHJ"),
    (3, "five-run attack, white to move", "v1 min-max"): (14461, 1577, "HJBJ"),
    (3, "five-run attack, white to move", "v2 alpha-beta"): (249, 1577, "HJBJ"),
    (3, "five-run attack, white to move", "v3 alpha-beta + tuned weights"): (249, 1536, "HJBJ"),
    (3, "five-run attack, white to move", "v4 + quiescence, pruning, book"): (103, 20000, "HJBJ"),
    (3, "five-run attack, white to move", "v5 + forced-win layer"): (104, 20000, "HJBJ"),
    (3, "five-run attack, white to move", "v6 + parallel split, budget"): (104, 20000, "HJBJ"),
    (4, "opening reply, white to move", "v2 alpha-beta"): (24533, 303, "IJJK"),
    (4, "opening reply, white to move", "v3 alpha-beta + tuned weights"): (25469, 282, "IJJK"),
    (4, "opening reply, white to move", "v4 + quiescence, pruning, book"): (0, None, "IIKK"),
    (4, "opening reply, white to move", "v5 + forced-win layer"): (0, None, "IIKK"),
    (4, "opening reply, white to move", "v6 + parallel split, budget"): (0, None, "IIKK"),
    (4, "opening scatter, white to move", "v2 alpha-beta"): (9150, -66, "KQCK"),
    (4, "opening scatter, white to move", "v3 alpha-beta + tuned weights"): (15964, -40, "CKJP"),
    (4, "opening scatter, white to move", "v4 + quiescence, pruning, book"): (7581, -38, "CKJP"),
    (4, "opening scatter, white to move", "v5 + forced-win layer"): (75026, -336, "IKKP"),
    (4, "opening scatter, white to move", "v6 + parallel split, budget"): (234146, -336, "IKKP"),
    (4, "midgame, white to move", "v2 alpha-beta"): (11640, -301, "FJKI"),
    (4, "midgame, white to move", "v3 alpha-beta + tuned weights"): (11799, -305, "FJKI"),
    (4, "midgame, white to move", "v4 + quiescence, pruning, book"): (3782, -617, "JGII"),
    (4, "midgame, white to move", "v5 + forced-win layer"): (21962, -1060, "JGKI"),
    (4, "midgame, white to move", "v6 + parallel split, budget"): (207618, -1060, "JGKI"),
    (4, "fork attack, black to move", "v2 alpha-beta"): (350, 20000, "JNBJ"),
    (4, "fork attack, black to move", "v3 alpha-beta + tuned weights"): (347, 20000, "JNBJ"),
    (4, "fork attack, black to move", "v4 + quiescence, pruning, book"): (179, 20000, "JNBJ"),
    (4, "fork attack, black to move", "v5 + forced-win layer"): (18, 20000, "JSFJ"),
    (4, "fork attack, black to move", "v6 + parallel split, budget"): (18, 20000, "JSFJ"),
    (4, "fork completed, white to move", "v2 alpha-beta"): (577, 20000, "GJHJ"),
    (4, "fork completed, white to move", "v3 alpha-beta + tuned weights"): (573, 20000, "GJHJ"),
    (4, "fork completed, white to move", "v4 + quiescence, pruning, book"): (349, 20000, "GJHJ"),
    (4, "fork completed, white to move", "v5 + forced-win layer"): (350, 20000, "GJHJ"),
    (4, "fork completed, white to move", "v6 + parallel split, budget"): (350, 20000, "GJHJ"),
    (4, "five-run attack, white to move", "v2 alpha-beta"): (1616, 11613, "HJBJ"),
    (4, "five-run attack, white to move", "v3 alpha-beta + tuned weights"): (1616, 11596, "HJBJ"),
    (4, "five-run attack, white to move", "v4 + quiescence, pruning, book"): (110, 20000, "HJBJ"),
    (4, "five-run attack, white to move", "v5 + forced-win layer"): (111, 20000, "HJBJ"),
    (4, "five-run attack, white to move", "v6 + parallel split, budget"): (111, 20000, "HJBJ"),
}

for key, expected in EXPECTED_COST.items():
    measured = cost_rows[key]
    assert measured == expected, f"{key}: measured {measured}, documented {expected}"
print(f"all {len(EXPECTED_COST)} cost rows reproduce section 12.2 exactly (nodes, score, move)")


## Charts


In [ ]:
import matplotlib.pyplot as plt

labels = [label for label, *_ in positions]
short = [label.split(",")[0].replace("opening ", "") for label in labels]
width = 0.15
fig, ax = plt.subplots(figsize=(10, 4.5))
for i, version in enumerate(VERSIONS):
    nodes = [max(cost_rows[(3, label, version.name)][0], 1) for label in labels]
    ax.bar([j + (i - 2.5) * width for j in range(len(labels))], nodes, width, label=version.name)
ax.set_yscale("log")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(short, fontsize=9)
ax.set_ylabel("nodes at depth 3 (log scale)")
ax.set_title("the cost ladder at the deployment depth (book hits clipped to 1 node)")
ax.grid(True, axis="y")
ax.legend(fontsize=8)
plt.show()


## Reading the results

- **v1 and v2 are the same player** — identical moves, ply 38, black wins both games of the
  pairing: the section-9 baseline mirror, reproduced.
- **Weights alone win games** — v3 beats v2 in both colors; **the enhancements carry it to
  full games** — v4 beats v3 in both colors.
- **v4 and v5 split by color** — white wins both games of their pairing on the book line:
  the layer fires only when a forced win exists, and until white's converting attack it
  prices exactly what the enhanced search prices.
- **v5 and v6 are the same player** — both games of the pairing are one game played twice,
  identical move for move: the parallel split changes the price of a move, never the move.
  Its node bill is the price of the full-window root (2-9x v5 per game here).
- **v6 beats v1 in both colors** on the full journey (19 and 30 plies) — the split root
  inherits the whole stack: book, tuned weights, quiescence and the forced-win layer.
- Full discussion: docs/EVALUATION.md, sections 12 and 13.
